#Homework 4: From Model to Production — End-to-End MLOps Pipeline

**PART 1 - Foundation Model Exploration**

In [ ]:
#Part 1A Setup and Installation
!pip install transformers torch
!pip install mflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib
import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Load Part 1 evaluation dataset - Went back to HW2 to save a new dataset
df_part1 = pd.read_csv('/content/drive/MyDrive/df_part1_eval.csv')
print("Dataset shape:", df_part1.shape)
print("Columns:", df_part1.columns.tolist())
df_part1.head()

Dataset shape: (98646, 16)
Columns: ['order_id', 'review_score', 'review_comment_message', 'is_positive_review', 'delivery_days', 'delivery_vs_estimated', 'freight_value', 'product_category_name', 'seller_state', 'payment_type_main', 'seller_historical_average_rating', 'is_new_seller', 'num_items', 'payment_value_total', 'order_hour', 'order_dayofweek']


,order_id,review_score,review_comment_message,is_positive_review,delivery_days,delivery_vs_estimated,freight_value,product_category_name,seller_state,payment_type_main,seller_historical_average_rating,is_new_seller,num_items,payment_value_total,order_hour,order_dayofweek
0,2e7a8482f6fb09756ca50c10d7bfc047,1.0,1 mes de atraso na entrega !!! ultima compra q...,0,NaN,NaN,31.67,furniture_decor,MG,credit_card,4.086265,1,2,136.23,21,Sunday
1,e5fa5a7210941f7d56d0208e4e071d35,1.0,Comprei dois produtos desta loja parceira da l...,0,NaN,NaN,15.56,telephony,PR,credit_card,4.086265,1,1,75.06,0,Monday
2,809a282bbd5dbcabb6f2f724fca862ec,1.0,MEU PEDIDO NÃO FOI ENTREGUE E NÃO FOI DADA NEN...,0,NaN,NaN,NaN,NaN,NaN,credit_card,4.086265,1,0,40.95,15,Tuesday
3,bfbd0f9bdef84302105ad712db648a6c,1.0,nao recebi o produto e nem resposta da empresa,0,54.0,36.0,2.83,health_beauty,PR,NaN,4.086265,1,3,NaN,12,Thursday
4,71303d7e93b399f5bcd537d124c0bcfa,1.0,No Message,0,NaN,NaN,9.34,baby,SP,credit_card,4.086265,1,1,109.34,22,Sunday


In [ ]:
#Model from HW2
model = joblib.load('/content/drive/MyDrive/Data6545/model.pkl')
print("Model Loaded Succesfully", type(model).__name__)

Model Loaded Succesfully Pipeline


In [ ]:
#Load the pre-trained multilingual sentiment model using HuggingFace's pipeline API:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model = "nlptown/bert-base-multilingual-uncased-sentiment")

print("Foundation model loaded")

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Foundation model loaded


In [ ]:
#Filter Olist dataset to records that  have non-empty review comment message text
df_part1['review_comment_message'] = df_part1['review_comment_message'].fillna('').astype(str)
reviews_with_text = df_part1[df_part1['review_comment_message'].str.strip() != ''].copy()

print(f"Total rows: {len(df_part1)}")
print(f"Rows with text: {len(reviews_with_text)}")

Total rows: 98646
Rows with text: 98646


In [ ]:
#Sample 500 records from this filtered set (use random state=42 for reproducibility).

sample = reviews_with_text.sample(n=500, random_state=42).reset_index(drop=True)
print(sample.shape)
sample[['order_id', 'review_score', 'review_comment_message', 'is_positive_review']].head()

(500, 16)


,order_id,review_score,review_comment_message,is_positive_review
0,8c698ded819cb4d7c0fa6be003d08768,1.0,o produto veio pela metade e veio outro produto,0
1,00063b381e2406b52ad429470734ebd5,5.0,"Fiquei um pouco triste, achei que a cor do cor...",1
2,7a7afad8caa7dffdb8773430c8fab7ea,4.0,No Message,1
3,d0d0beb416cef186261cf1f2287bef19,4.0,No Message,1
4,431eb716029df1431f4ff0fbe57d2f06,4.0,No Message,1


In [ ]:
print(f"Sample Size: {len(sample)}")
print(f"Positive Reviews: {len(sample[sample['is_positive_review'] == 1])}")
print(f"Negative Reviews: {len(sample[sample['is_positive_review'] == 0])}")

Sample Size: 500
Positive Reviews: 399
Negative Reviews: 101


In [ ]:
#Run the foundation model on each review comment. The model returns a label like "3 stars" and a confidence score.

foundation_preds = []
foundation_Scores = []

for i, text in enumerate(sample['review_comment_message']):
  result = sentiment(str(text), truncation = True, max_length = 512)[0]
  stars = int(result['label'].split()[0])
  binary = 1 if stars >= 4 else 0

  foundation_preds.append(binary)
  foundation_Scores.append(result['score'])

  if (i+1) % 100 ==0:
    print(f"  Processed {i+1}/500 reviews...")

sample['foundation_pred'] = foundation_preds
sample['foundation_score'] = foundation_Scores

print(sample['foundation_pred'].value_counts())

  Processed 100/500 reviews...
  Processed 200/500 reviews...
  Processed 300/500 reviews...
  Processed 400/500 reviews...
  Processed 500/500 reviews...
foundation_pred
0    384
1    116
Name: count, dtype: int64


In [ ]:
#Features for HW2 model

# Features
FEATURES = [
    'delivery_days',
    'delivery_vs_estimated',
    'freight_value',
    'product_category_name',
    'seller_state',
    'payment_type_main',
    'seller_historical_average_rating',
    'is_new_seller',
    'num_items',
    'payment_value_total',
    'order_hour',
    'order_dayofweek'
]

X_sample = sample[FEATURES]
y_true = sample['is_positive_review']

# Get HW2 model predictions on the same 500 records
hw2_preds = model.predict(X_sample)
hw2_probs = model.predict_proba(X_sample)[:, 1]

sample['hw2_pred'] = hw2_preds
sample['hw2_prob'] = hw2_probs

print(f"HW2 model predictions complete!")
print(f"Predicted positive: {hw2_preds.sum()}")
print(f"Predicted negative: {(hw2_preds == 0).sum()}")

HW2 model predictions complete!
Predicted positive: 404
Predicted negative: 96


In [ ]:
#Class distribution
print("Actual class distribution:")
print(sample['is_positive_review'].value_counts())

print("\nFoundation model predicted distribution:")
print(sample['foundation_pred'].value_counts())

print("\nHW2 model predicted distribution:")
print(sample['hw2_pred'].value_counts())

Actual class distribution:
is_positive_review
1    399
0    101
Name: count, dtype: int64

Foundation model predicted distribution:
foundation_pred
0    384
1    116
Name: count, dtype: int64

HW2 model predicted distribution:
hw2_pred
1    404
0     96
Name: count, dtype: int64


In [ ]:
#1B Performance Comparison
hw2_accuracy = accuracy_score(y_true, hw2_preds)
hw2_precision = precision_score(y_true, hw2_preds)
hw2_recall = recall_score(y_true, hw2_preds)
hw2_f1 = f1_score(y_true, hw2_preds)

print("HW2 (LightGBM) Model Metrics:")
print(f"HW2 Accuracy: {hw2_accuracy:.4f}")
print(f"HW2 Precision: {hw2_precision:.4f}")
print(f"HW2 Recall: {hw2_recall:.4f}")
print(f"HW2 F1 Score: {hw2_f1:.4f}")

HW2 (LightGBM) Model Metrics:
HW2 Accuracy: 0.7780
HW2 Precision: 0.8564
HW2 Recall: 0.8672
HW2 F1 Score: 0.8618


In [ ]:
#Metrics for Foundation Model
foundation_accuracy = accuracy_score(y_true, sample['foundation_pred'])
foundation_precision = precision_score(y_true, sample['foundation_pred'])
foundation_recall = recall_score(y_true, sample['foundation_pred'])
foundation_f1 = f1_score(y_true, sample['foundation_pred'])

print("Foundation Model Metrics:")
print(f" Accuracy: {foundation_accuracy:.4f}")
print(f" Precision: {foundation_precision:.4f}")
print(f" Recall: {foundation_recall:.4f}")
print(f" F1 Score: {foundation_f1:.4f}")

Foundation Model Metrics:
 Accuracy: 0.4260
 Precision: 0.9828
 Recall: 0.2857
 F1 Score: 0.4427


In [ ]:
# Comparison Table
comparison = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Foundation (Review Text)': [
        foundation_accuracy,
        foundation_precision,
        foundation_recall,
        foundation_f1
    ],
    'HW2 Light GBM (Order Features)': [
        hw2_accuracy,
        hw2_precision,
        hw2_recall,
        hw2_f1
    ]
}

comparison_df = pd.DataFrame(comparison)
comparison_df.set_index('Metric', inplace=True)
print(comparison_df)
comparison_df

           Foundation (Review Text)  HW2 Light GBM (Order Features)
Metric                                                             
Accuracy                   0.426000                        0.778000
Precision                  0.982759                        0.856436
Recall                     0.285714                        0.867168
F1 Score                   0.442718                        0.861768


,Foundation (Review Text),HW2 Light GBM (Order Features)
Metric,,
Accuracy,0.426000,0.778000
Precision,0.982759,0.856436
Recall,0.285714,0.867168
F1 Score,0.442718,0.861768


**Reflection**

The HuggingFace foundation model was very interesting to explore! Overall, the model that performed best on these 500 records was the HW 2 Tuned LightGBM model. This model seems to be the consistent winner across all metrics: accuracy(0.77 vs 0.42),
recall (0.86 vs 0.28), and F1 score (0.86 vs 0.44).The foundational model did achielve a high precision score of 0.98 but it predicted far fewer positive reviews, which caused its recall to also be lower.

I believe LightGBM performed better in this scenario because it was specifically trained Olist order and delivery data, so it learned patterns on how to identify a negative review. The foundation model was not trained the same way and was analyzing review text reactively, after the customer experience. Our business goal is proactive prediction before the review is written, which is where the LightGBM model does best.

Since the foundation model had zero training data, a big advantage was that the model was ready much quicker. However, it did not perform as well as the LightGBM model, which was trained.

I would recommend using a foundational model when you have a lot of text analysis involved or when you need a quick solution with no labeled training data. i would recommend training a custom model when you have a specific goal, like in our case, it is to identify dissatisfied customers before they leave a negative review.

Finally, I also believe these two approaches could be combined in a production system! LightGBM could flag the at risk customers and the foundational model could analyze review text after submission. They both are strong in different ways and could compliment each other.

**PART 2 - Experiment Tracking with MLFlow**

In [ ]:
!pip install mlflow
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.5/857.5 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.

In [ ]:
#Create a new MLflow experiment called "olist-satisfaction"
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("olist-satisfaction")

print(f"Tracking URI: {mlflow.get_tracking_uri()}")

2026/04/22 00:38:18 INFO mlflow.tracking.fluent: Experiment with name 'olist-satisfaction' does not exist. Creating a new experiment.


Tracking URI: mlruns


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score


In [ ]:
df_model = pd.read_csv('/content/drive/MyDrive/df_model.csv')
print(df_model.shape)
df_model.head()

(98673, 14)


,delivery_days,delivery_vs_estimated,freight_value,product_category_name,seller_state,payment_type_main,seller_historical_average_rating,is_new_seller,num_items,payment_value_total,order_hour,order_dayofweek,is_positive_review,freight_ratio
0,NaN,NaN,31.67,furniture_decor,MG,credit_card,4.086265,1,2,136.23,21,Sunday,0,0.232474
1,NaN,NaN,15.56,telephony,PR,credit_card,4.086265,1,1,75.06,0,Monday,0,0.207301
2,NaN,NaN,NaN,NaN,NaN,credit_card,4.086265,1,0,40.95,15,Tuesday,0,NaN
3,54.0,36.0,2.83,health_beauty,PR,NaN,4.086265,1,3,NaN,12,Thursday,0,NaN
4,NaN,NaN,9.34,baby,SP,credit_card,4.086265,1,1,109.34,22,Sunday,0,0.085422


In [ ]:
#Log at least 2 model training runs from HW 2 or 3. for each, log parameteres, metrics, artifact
#Log 1
features =  ['delivery_days', 'delivery_vs_estimated', 'freight_value',
            'product_category_name', 'seller_state', 'payment_type_main',
            'seller_historical_average_rating', 'is_new_seller',
            'num_items', 'payment_value_total', 'order_hour', 'order_dayofweek']

X = df_model[features]
y = df_model['is_positive_review']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)

numeric_features = ['delivery_days', 'delivery_vs_estimated', 'freight_value',
                    'seller_historical_average_rating', 'is_new_seller',
                    'num_items', 'payment_value_total', 'order_hour']
categorical_features = ['product_category_name', 'seller_state',
                        'payment_type_main', 'order_dayofweek']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_features)
])

# Train Logistic Regression
with mlflow.start_run(run_name="logistic-regression-baseline"):
    lr = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_test)
    y_prob = lr.predict_proba(X_test)[:, 1]

    # Log parameters
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("accuracy", round(accuracy_score(y_test, y_pred), 4))
    mlflow.log_metric("precision", round(precision_score(y_test, y_pred), 4))
    mlflow.log_metric("recall", round(recall_score(y_test, y_pred), 4))
    mlflow.log_metric("f1", round(f1_score(y_test, y_pred), 4))
    mlflow.log_metric("roc_auc", round(roc_auc_score(y_test, y_prob), 4))


    mlflow.sklearn.log_model(lr, "model")

    print("Run 1 - Logistic Regression")
    print(f"  Accuracy: {round(accuracy_score(y_test, y_pred), 4)}")
    print(f"  Precision: {round(precision_score(y_test, y_pred), 4)}")
    print(f"  Recall: {round(recall_score(y_test, y_pred), 4)}")
    print(f"  F1: {round(f1_score(y_test, y_pred), 4)}")
    print(f"  ROC AUC: {round(roc_auc_score(y_test, y_prob), 4)}")


2026/04/22 00:39:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 00:39:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 1 - Logistic Regression
  Accuracy: 0.7965
  Precision: 0.8
  Recall: 0.9811
  F1: 0.8814
  ROC AUC: 0.6933


In [ ]:
#Log2
from lightgbm import LGBMClassifier

with mlflow.start_run(run_name="tuned-lightgbm"):
    lgbm = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LGBMClassifier(
            n_estimators=100,
            random_state=42,
            verbose=-1
        ))
    ])
    lgbm.fit(X_train, y_train)

    y_pred = lgbm.predict(X_test)
    y_prob = lgbm.predict_proba(X_test)[:, 1]

    # Log parameters
    mlflow.log_param("model_type", "LightGBM")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)

    # Log metrics
    mlflow.log_metric("accuracy", round(accuracy_score(y_test, y_pred), 4))
    mlflow.log_metric("precision", round(precision_score(y_test, y_pred), 4))
    mlflow.log_metric("recall", round(recall_score(y_test, y_pred), 4))
    mlflow.log_metric("f1", round(f1_score(y_test, y_pred), 4))
    mlflow.log_metric("roc_auc", round(roc_auc_score(y_test, y_prob), 4))

    # Log model artifact
    mlflow.sklearn.log_model(lgbm, "model")

    print("Run 2 - LightGBM")
    print(f"  Accuracy: {round(accuracy_score(y_test, y_pred), 4)}")
    print(f"  Precision: {round(precision_score(y_test, y_pred), 4)}")
    print(f"  Recall: {round(recall_score(y_test, y_pred), 4)}")
    print(f"  F1: {round(f1_score(y_test, y_pred), 4)}")
    print(f"  ROC AUC: {round(roc_auc_score(y_test, y_prob), 4)}")


2026/04/22 00:39:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 00:39:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 2 - LightGBM
  Accuracy: 0.8233
  Precision: 0.8273
  Recall: 0.974
  F1: 0.8947
  ROC AUC: 0.7437


In [ ]:
#All logged tuns

client = MlflowClient()
experiment = client.get_experiment_by_name("olist-satisfaction")
runs = client.search_runs(
    experiment_ids= [experiment.experiment_id],
    order_by=["metrics.roc_auc DESC"]
)

print("All logged runs:")
for run in runs:
    print(f"  Run ID: {run.info.run_id}")
    print(f"Model Type: {run.data.params.get('model_type')}")
    print(f"  Accuracy: {run.data.metrics.get('accuracy')}")
    print(f"  Precision: {run.data.metrics.get('precision')}")
    print(f"  Recall: {run.data.metrics.get('recall')}")
    print(f"  F1: {run.data.metrics.get('f1')}")
    print(f"  ROC AUC: {run.data.metrics.get('roc_auc')}")
    print()

All logged runs:
  Run ID: 8215a60064df42f38c6a4507b0d6d4ff
Model Type: LightGBM
  Accuracy: 0.8233
  Precision: 0.8273
  Recall: 0.974
  F1: 0.8947
  ROC AUC: 0.7437

  Run ID: 3e57133dbac74c7a9937cdd8e29a3eba
Model Type: LogisticRegression
  Accuracy: 0.7965
  Precision: 0.8
  Recall: 0.9811
  F1: 0.8814
  ROC AUC: 0.6933



In [ ]:
# Save mlruns folder to Google Drive
import shutil

shutil.rmtree('/content/drive/MyDrive/mlruns')
shutil.copytree('mlruns', '/content/drive/MyDrive/mlruns')

print("mlruns saved!")

mlruns saved!


In [ ]:
#Register best model
best_run_id = "8215a60064df42f38c6a4507b0d6d4ff"
model_uri = f"runs:/{best_run_id}/model"
registered_model = mlflow.register_model(model_uri, "olist-satisfaction-model")

print("Model Registered")
print(f"Name: {registered_model.name}")
print(f"Version: {registered_model.version}")
print(f"Run ID: {registered_model.run_id}")

Registered model 'olist-satisfaction-model' already exists. Creating a new version of this model...
2026/04/22 00:47:21 WARNING mlflow.tracking._model_registry.fluent: Run with id 8215a60064df42f38c6a4507b0d6d4ff has no artifacts at artifact path 'model', registering model based on models:/m-a3ddeca10f4e4610b6fba1e3273fe4a9 instead


Model Registered
Name: olist-satisfaction-model
Version: 1
Run ID: 8215a60064df42f38c6a4507b0d6d4ff


Created version '1' of model 'olist-satisfaction-model'.


In [ ]:
#Transition model to "Production" stage
client.transition_model_version_stage(
    name = "olist-satisfaction-model",
    version = 1,
    stage = "Production"
)

print("Model Transitioned to Production")
print("Model: olist-satisfaction-model")
print("Version: 1")
print("Stage: Production")

Model Transitioned to Production
Model: olist-satisfaction-model
Version: 1
Stage: Production


In [ ]:
import shutil
import os
if os.path.exists('/content/drive/MyDrive/mlruns'):
    shutil.rmtree('/content/drive/MyDrive/mlruns')
shutil.copytree('mlruns', '/content/drive/MyDrive/mlruns')
print("mlruns updated in Google Drive!")

mlruns updated in Google Drive!
